In [124]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

print("Current Notebook :", Path.cwd())
print("Project Root     :", PROJECT_ROOT)

print("\nExists?")
print(PROJECT_ROOT.exists())

print("\nContents:")
for item in PROJECT_ROOT.iterdir():
    print(item.name)

Current Notebook : c:\Users\divya\Downloads\Retail-Intelligence-Platform\notebooks
Project Root     : c:\Users\divya\Downloads\Retail-Intelligence-Platform

Exists?
True

Contents:
.gitignore
dashboard_exports
data
documentation
images
LICENSE
notebooks
powerbi
presentation
README.md
reports
requirements.txt
src
tests


In [125]:
# ======================================================
# Project Paths
# ======================================================

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

DATA_FOLDER = PROJECT_ROOT / "data"

# Create the folder if it doesn't exist
DATA_FOLDER.mkdir(parents=True, exist_ok=True)

RAW_DATA = DATA_FOLDER / "raw"

DATABASE = DATA_FOLDER / "retail.db"

print("Project Root :", PROJECT_ROOT)
print("Data Folder  :", DATA_FOLDER)
print("Raw Folder   :", RAW_DATA)
print("Database     :", DATABASE)

Project Root : c:\Users\divya\Downloads\Retail-Intelligence-Platform
Data Folder  : c:\Users\divya\Downloads\Retail-Intelligence-Platform\data
Raw Folder   : c:\Users\divya\Downloads\Retail-Intelligence-Platform\data\raw
Database     : c:\Users\divya\Downloads\Retail-Intelligence-Platform\data\retail.db


In [126]:
from pathlib import Path
import os

print("="*60)

print("Current Working Directory")
print(Path.cwd())

print("\nCurrent Directory Exists?")
print(Path.cwd().exists())

print("\nParent")
print(Path.cwd().parent)

print("\nProject Structure")

for item in Path.cwd().iterdir():
    print(item)

print("="*60)

Current Working Directory
c:\Users\divya\Downloads\Retail-Intelligence-Platform\notebooks

Current Directory Exists?
True

Parent
c:\Users\divya\Downloads\Retail-Intelligence-Platform

Project Structure
c:\Users\divya\Downloads\Retail-Intelligence-Platform\notebooks\01_Data_Understanding.ipynb
c:\Users\divya\Downloads\Retail-Intelligence-Platform\notebooks\02_Data_Cleaning.ipynb
c:\Users\divya\Downloads\Retail-Intelligence-Platform\notebooks\03_Exploratory_Data_Analysis.ipynb
c:\Users\divya\Downloads\Retail-Intelligence-Platform\notebooks\04_SQL_Business_Analysis.ipynb
c:\Users\divya\Downloads\Retail-Intelligence-Platform\notebooks\05_Feature_Engineering.ipynb
c:\Users\divya\Downloads\Retail-Intelligence-Platform\notebooks\06_Sales_Forecasting.ipynb
c:\Users\divya\Downloads\Retail-Intelligence-Platform\notebooks\07_Customer_Segmentation_RFM.ipynb
c:\Users\divya\Downloads\Retail-Intelligence-Platform\notebooks\08_Market_Basket_Analysis.ipynb
c:\Users\divya\Downloads\Retail-Intelligence-

In [127]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

print(PROJECT_ROOT)

print((PROJECT_ROOT / "data").exists())
print((PROJECT_ROOT / "data" / "raw").exists())

c:\Users\divya\Downloads\Retail-Intelligence-Platform\notebooks
False
False


In [128]:
%pip install duckdb


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [129]:
import duckdb

con = duckdb.connect()

print("DuckDB Connected Successfully")

DuckDB Connected Successfully


In [130]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RAW_DATA = PROJECT_ROOT / "data" / "raw"

datasets = {}

for file in RAW_DATA.glob("*.csv"):

    datasets[file.stem] = pd.read_csv(file)

print(f"Loaded {len(datasets)} datasets.")

Loaded 9 datasets.


In [131]:
for name, df in datasets.items():

    con.register(name, df)

print("All DataFrames registered successfully.")

All DataFrames registered successfully.


In [132]:
con.sql("SHOW TABLES").df()

,name
0,olist_customers_dataset
1,olist_geolocation_dataset
2,olist_order_items_dataset
3,olist_order_payments_dataset
4,olist_order_reviews_dataset
5,olist_orders_dataset
6,olist_products_dataset
7,olist_sellers_dataset
8,product_category_name_translation


In [133]:
con.sql("""

SELECT

ROUND(SUM(payment_value),2) AS TotalRevenue

FROM olist_order_payments_dataset

""").df()

,TotalRevenue
0,16008872.12


In [134]:
con.sql("""

SELECT

COUNT(DISTINCT order_id) AS TotalOrders

FROM olist_orders_dataset

""").df()

,TotalOrders
0,99441


In [135]:
con.sql("""

SELECT

COUNT(DISTINCT customer_unique_id) AS TotalCustomers

FROM olist_customers_dataset

""").df()

,TotalCustomers
0,96096


In [136]:
# ======================================================
# Revenue by Payment Type
# ======================================================

con.sql("""

SELECT

    payment_type,

    COUNT(*) AS TotalPayments,

    ROUND(SUM(payment_value),2) AS Revenue,

    ROUND(AVG(payment_value),2) AS AveragePayment

FROM olist_order_payments_dataset

GROUP BY payment_type

ORDER BY Revenue DESC;

""").df()

,payment_type,TotalPayments,Revenue,AveragePayment
0,credit_card,76795,12542084.19,163.32
1,boleto,19784,2869361.27,145.03
2,voucher,5775,379436.87,65.70
3,debit_card,1529,217989.79,142.57
4,not_defined,3,0.00,0.00


In [137]:
# ======================================================
# Order Status Distribution
# ======================================================

con.sql("""

SELECT

    order_status,

    COUNT(*) AS Orders

FROM olist_orders_dataset

GROUP BY order_status

ORDER BY Orders DESC;

""").df()

,order_status,Orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [138]:
# ======================================================
# Monthly Orders
# ======================================================

con.sql("""

SELECT

    strftime(

        CAST(order_purchase_timestamp AS TIMESTAMP),

        '%Y-%m'

    ) AS Month,

    COUNT(order_id) AS Orders

FROM olist_orders_dataset

GROUP BY Month

ORDER BY Month;

""").df()

,Month,Orders
0,2016-09,4
1,2016-10,324
2,2016-12,1
3,2017-01,800
4,2017-02,1780
5,2017-03,2682
6,2017-04,2404
7,2017-05,3700
8,2017-06,3245
9,2017-07,4026


In [139]:
# ======================================================
# Monthly Revenue
# ======================================================

con.sql("""

SELECT

    strftime(

        CAST(o.order_purchase_timestamp AS TIMESTAMP),

        '%Y-%m'

    ) AS Month,

    ROUND(

        SUM(p.payment_value),

        2

    ) AS Revenue

FROM olist_orders_dataset o

JOIN olist_order_payments_dataset p

ON o.order_id = p.order_id

GROUP BY Month

ORDER BY Month;

""").df()

,Month,Revenue
0,2016-09,252.24
1,2016-10,59090.48
2,2016-12,19.62
3,2017-01,138488.04
4,2017-02,291908.01
5,2017-03,449863.60
6,2017-04,417788.03
7,2017-05,592918.82
8,2017-06,511276.38
9,2017-07,592382.92


In [140]:
# ======================================================
# Top Customer States
# ======================================================

con.sql("""

SELECT

    customer_state,

    COUNT(DISTINCT customer_unique_id) AS Customers

FROM olist_customers_dataset

GROUP BY customer_state

ORDER BY Customers DESC

LIMIT 10;

""").df()

,customer_state,Customers
0,SP,40302
1,RJ,12384
2,MG,11259
3,RS,5277
4,PR,4882
5,SC,3534
6,BA,3277
7,DF,2075
8,ES,1964
9,GO,1952


In [141]:
# ======================================================
# Top Product Categories
# ======================================================

con.sql("""

SELECT

    t.product_category_name_english,

    COUNT(*) AS Orders,

    ROUND(SUM(i.price),2) AS Revenue

FROM olist_order_items_dataset i

JOIN olist_products_dataset p

ON i.product_id=p.product_id

JOIN product_category_name_translation t

ON p.product_category_name=t.product_category_name

GROUP BY t.product_category_name_english

ORDER BY Revenue DESC

LIMIT 10;

""").df()

,product_category_name_english,Orders,Revenue
0,health_beauty,9670,1258681.34
1,watches_gifts,5991,1205005.68
2,bed_bath_table,11115,1036988.68
3,sports_leisure,8641,988048.97
4,computers_accessories,7827,911954.32
5,furniture_decor,8334,729762.49
6,cool_stuff,3796,635290.85
7,housewares,6964,632248.66
8,auto,4235,592720.11
9,garden_tools,4347,485256.46


In [142]:
# ======================================================
# Average Order Value
# ======================================================

con.sql("""

SELECT

ROUND(

AVG(OrderValue),2

) AS AverageOrderValue

FROM

(

SELECT

order_id,

SUM(payment_value) AS OrderValue

FROM olist_order_payments_dataset

GROUP BY order_id

);

""").df()

,AverageOrderValue
0,160.99


In [143]:
# ======================================================
# Highest Value Orders
# ======================================================

con.sql("""

SELECT

order_id,

ROUND(

SUM(payment_value),2

) AS Revenue

FROM olist_order_payments_dataset

GROUP BY order_id

ORDER BY Revenue DESC

LIMIT 20;

""").df()

,order_id,Revenue
0,03caa2c082116e1d31e67e9ae3700499,13664.08
1,736e1922ae60d0d6a89247b851902527,7274.88
2,0812eb902a67711a1cb742b3cdaa65ae,6929.31
3,fefacc66af859508bf1a7934eab1e97f,6922.21
4,f5136e38d1a14a4dbd87dff67da82701,6726.66
5,2cc9089445046817a7539d90805e6e5a,6081.54
6,a96610ab360d42a2e5335a3998b4718a,4950.34
7,b4c4b76c642808cbe472a32b86cddc95,4809.44
8,199af31afc78c699f0dbf71fb178d4d4,4764.34
9,8dbc85d1447242f3b127dda390d56e19,4681.78


In [144]:
# ======================================================
# Top Sellers
# ======================================================

con.sql("""

SELECT

seller_id,

COUNT(*) AS Orders,

ROUND(

SUM(price),2

) AS Revenue

FROM olist_order_items_dataset

GROUP BY seller_id

ORDER BY Revenue DESC

LIMIT 20;

""").df()

,seller_id,Orders,Revenue
0,4869f7a5dfa277a7dca6462dcf3b52b2,1156,229472.63
1,53243585a1d6dc2643021fd1853d8905,410,222776.05
2,4a3ca9315b744ce9f8e9374361493884,1987,200472.92
3,fa1c13f2614d7b5c4749cbc52fecda94,586,194042.03
4,7c67e1448b00f6e969d365cea6b010ab,1364,187923.89
5,7e93a43ef30c4f03f38b393420bc753a,340,176431.87
6,da8622b14eb17ae2831f4ac5b9dab84a,1551,160236.57
7,7a67c85e85bb2ce8582c35f2203ad736,1171,141745.53
8,1025f0e2d44d7041d6cf58b6550e0bfa,1428,138968.55
9,955fee9216a65b617aa5c0531780ce60,1499,135171.70


In [145]:
# ======================================================
# Executive SQL Dashboard
# ======================================================

dashboard = con.sql("""

SELECT

(

SELECT ROUND(SUM(payment_value),2)

FROM olist_order_payments_dataset

) AS TotalRevenue,

(

SELECT COUNT(DISTINCT order_id)

FROM olist_orders_dataset

) AS Orders,

(

SELECT COUNT(DISTINCT customer_unique_id)

FROM olist_customers_dataset

) AS Customers,

(

SELECT ROUND(AVG(review_score),2)

FROM olist_order_reviews_dataset

) AS AverageReview;

""").df()

display(dashboard)

,TotalRevenue,Orders,Customers,AverageReview
0,16008872.12,99441,96096,4.09


In [146]:
# ======================================================
# Top Customers by Revenue
# ======================================================

con.sql("""

SELECT

    c.customer_unique_id,

    ROUND(
        SUM(p.payment_value),
        2
    ) AS Revenue

FROM olist_customers_dataset c

JOIN olist_orders_dataset o

ON c.customer_id = o.customer_id

JOIN olist_order_payments_dataset p

ON o.order_id = p.order_id

GROUP BY c.customer_unique_id

ORDER BY Revenue DESC

LIMIT 20;

""").df()

,customer_unique_id,Revenue
0,0a0a92112bd4c708ca5fde585afaa872,13664.08
1,46450c74a0d8c5ca9395da1daac6c120,9553.02
2,da122df9eeddfedc1dc1f5349a1a690c,7571.63
3,763c8b1c9c68a0229c42c9fc6f662b93,7274.88
4,dc4802a71eae9be1dd28f5d788ceb526,6929.31
5,459bef486812aa25204be022145caa62,6922.21
6,ff4159b92c40ebe40454e3e6a7c35ed6,6726.66
7,4007669dec559734d6f53e029e360987,6081.54
8,5d0a2980b292d049061542014e8960bf,4809.44
9,eebb5dda148d3893cdaf5b5ca3040ccb,4764.34


In [147]:
# ======================================================
# Customer Lifetime Value
# ======================================================

con.sql("""

WITH customer_revenue AS (

SELECT

    c.customer_unique_id,

    SUM(p.payment_value) AS Revenue

FROM olist_customers_dataset c

JOIN olist_orders_dataset o

ON c.customer_id = o.customer_id

JOIN olist_order_payments_dataset p

ON o.order_id = p.order_id

GROUP BY c.customer_unique_id

)

SELECT

ROUND(
    AVG(Revenue),
    2
) AS Average_CLV

FROM customer_revenue;

""").df()

,Average_CLV
0,166.59


In [148]:
# ======================================================
# Seller Ranking
# ======================================================

con.sql("""

SELECT

seller_id,

ROUND(

SUM(price),

2

) AS Revenue,

RANK() OVER(

ORDER BY SUM(price) DESC

) AS SellerRank

FROM olist_order_items_dataset

GROUP BY seller_id

LIMIT 20;

""").df()

,seller_id,Revenue,SellerRank
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1
1,53243585a1d6dc2643021fd1853d8905,222776.05,2
2,4a3ca9315b744ce9f8e9374361493884,200472.92,3
3,fa1c13f2614d7b5c4749cbc52fecda94,194042.03,4
4,7c67e1448b00f6e969d365cea6b010ab,187923.89,5
5,7e93a43ef30c4f03f38b393420bc753a,176431.87,6
6,da8622b14eb17ae2831f4ac5b9dab84a,160236.57,7
7,7a67c85e85bb2ce8582c35f2203ad736,141745.53,8
8,1025f0e2d44d7041d6cf58b6550e0bfa,138968.55,9
9,955fee9216a65b617aa5c0531780ce60,135171.70,10


In [149]:
# ======================================================
# Dense Rank
# ======================================================

con.sql("""

SELECT

seller_id,

ROUND(

SUM(price),

2

) AS Revenue,

DENSE_RANK() OVER(

ORDER BY SUM(price) DESC

) AS DenseRank

FROM olist_order_items_dataset

GROUP BY seller_id

LIMIT 20;

""").df()

,seller_id,Revenue,DenseRank
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1
1,53243585a1d6dc2643021fd1853d8905,222776.05,2
2,4a3ca9315b744ce9f8e9374361493884,200472.92,3
3,fa1c13f2614d7b5c4749cbc52fecda94,194042.03,4
4,7c67e1448b00f6e969d365cea6b010ab,187923.89,5
5,7e93a43ef30c4f03f38b393420bc753a,176431.87,6
6,da8622b14eb17ae2831f4ac5b9dab84a,160236.57,7
7,7a67c85e85bb2ce8582c35f2203ad736,141745.53,8
8,1025f0e2d44d7041d6cf58b6550e0bfa,138968.55,9
9,955fee9216a65b617aa5c0531780ce60,135171.70,10


In [150]:
# ======================================================
# Category Ranking
# ======================================================

con.sql("""

SELECT

t.product_category_name_english,

ROUND(

SUM(i.price),

2

) AS Revenue,

RANK() OVER(

ORDER BY SUM(i.price) DESC

) AS CategoryRank

FROM olist_order_items_dataset i

JOIN olist_products_dataset p

ON i.product_id=p.product_id

JOIN product_category_name_translation t

ON p.product_category_name=t.product_category_name

GROUP BY t.product_category_name_english

LIMIT 20;

""").df()

,product_category_name_english,Revenue,CategoryRank
0,health_beauty,1258681.34,1
1,watches_gifts,1205005.68,2
2,bed_bath_table,1036988.68,3
3,sports_leisure,988048.97,4
4,computers_accessories,911954.32,5
5,furniture_decor,729762.49,6
6,cool_stuff,635290.85,7
7,housewares,632248.66,8
8,auto,592720.11,9
9,garden_tools,485256.46,10


In [151]:
# ======================================================
# Running Revenue
# ======================================================

con.sql("""

WITH MonthlyRevenue AS (

SELECT

date_trunc(

'month',

CAST(o.order_purchase_timestamp AS TIMESTAMP)

) AS Month,

SUM(p.payment_value) AS Revenue

FROM olist_orders_dataset o

JOIN olist_order_payments_dataset p

ON o.order_id = p.order_id

GROUP BY 1

)

SELECT

Month,

Revenue,

SUM(Revenue)

OVER(

ORDER BY Month

ROWS BETWEEN UNBOUNDED PRECEDING

AND CURRENT ROW

) AS RunningRevenue

FROM MonthlyRevenue

ORDER BY Month;

""").df()

,Month,Revenue,RunningRevenue
0,2016-09-01,252.24,252.24
1,2016-10-01,59090.48,59342.72
2,2016-12-01,19.62,59362.34
3,2017-01-01,138488.04,197850.38
4,2017-02-01,291908.01,489758.39
5,2017-03-01,449863.60,939621.99
6,2017-04-01,417788.03,1357410.02
7,2017-05-01,592918.82,1950328.84
8,2017-06-01,511276.38,2461605.22
9,2017-07-01,592382.92,3053988.14


In [152]:
# ======================================================
# Monthly Revenue Growth
# ======================================================

con.sql("""

WITH MonthlyRevenue AS (

SELECT

date_trunc(

'month',

CAST(o.order_purchase_timestamp AS TIMESTAMP)

) AS Month,

SUM(p.payment_value) AS Revenue

FROM olist_orders_dataset o

JOIN olist_order_payments_dataset p

ON o.order_id = p.order_id

GROUP BY 1

)

SELECT

Month,

Revenue,

LAG(Revenue)

OVER(

ORDER BY Month

) AS PreviousRevenue,

ROUND(

100.0 *

(

Revenue

-

LAG(Revenue)

OVER(ORDER BY Month)

)

/

LAG(Revenue)

OVER(ORDER BY Month),

2

) AS GrowthPercent

FROM MonthlyRevenue

ORDER BY Month;

""").df()

,Month,Revenue,PreviousRevenue,GrowthPercent
0,2016-09-01,252.24,NaN,NaN
1,2016-10-01,59090.48,252.24,23326.29
2,2016-12-01,19.62,59090.48,-99.97
3,2017-01-01,138488.04,19.62,705751.38
4,2017-02-01,291908.01,138488.04,110.78
5,2017-03-01,449863.60,291908.01,54.11
6,2017-04-01,417788.03,449863.60,-7.13
7,2017-05-01,592918.82,417788.03,41.92
8,2017-06-01,511276.38,592918.82,-13.77
9,2017-07-01,592382.92,511276.38,15.86


In [153]:
con.sql("""

DESCRIBE olist_orders_dataset

""").df()

,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,customer_id,VARCHAR,YES,None,None,None
2,order_status,VARCHAR,YES,None,None,None
3,order_purchase_timestamp,VARCHAR,YES,None,None,None
4,order_approved_at,VARCHAR,YES,None,None,None
5,order_delivered_carrier_date,VARCHAR,YES,None,None,None
6,order_delivered_customer_date,VARCHAR,YES,None,None,None
7,order_estimated_delivery_date,VARCHAR,YES,None,None,None


In [154]:
# ======================================================
# Top 3 Sellers
# ======================================================

con.sql("""

SELECT *

FROM (

SELECT

seller_id,

SUM(price) AS Revenue,

ROW_NUMBER()

OVER(

ORDER BY SUM(price) DESC

) AS rn

FROM olist_order_items_dataset

GROUP BY seller_id

)

WHERE rn<=3;

""").df()

,seller_id,Revenue,rn
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1
1,53243585a1d6dc2643021fd1853d8905,222776.05,2
2,4a3ca9315b744ce9f8e9374361493884,200472.92,3


In [155]:
# ======================================================
# Highest Value Orders
# ======================================================

con.sql("""

SELECT

order_id,

ROUND(

SUM(payment_value),

2

) AS Revenue

FROM olist_order_payments_dataset

GROUP BY order_id

HAVING SUM(payment_value)>1000

ORDER BY Revenue DESC;

""").df()

,order_id,Revenue
0,03caa2c082116e1d31e67e9ae3700499,13664.08
1,736e1922ae60d0d6a89247b851902527,7274.88
2,0812eb902a67711a1cb742b3cdaa65ae,6929.31
3,fefacc66af859508bf1a7934eab1e97f,6922.21
4,f5136e38d1a14a4dbd87dff67da82701,6726.66
...,...,...
1169,c40cbf2797c3b0187b41ea072dcac55f,1003.42
1170,491fc1fb624207b2123368e6b0ea4df1,1003.08
1171,fa794cef99ed64f42ebb766a8af51c64,1002.79
1172,62d9b911d7c56cf455f660eecb8ddd3a,1002.73


In [156]:
# ======================================================
# SQL Executive Dashboard
# ======================================================

dashboard = con.sql("""

SELECT

(

SELECT ROUND(SUM(payment_value),2)

FROM olist_order_payments_dataset

) AS Revenue,

(

SELECT COUNT(DISTINCT order_id)

FROM olist_orders_dataset

) AS Orders,

(

SELECT COUNT(DISTINCT customer_unique_id)

FROM olist_customers_dataset

) AS Customers,

(

SELECT ROUND(AVG(review_score),2)

FROM olist_order_reviews_dataset

) AS AvgReview,

(

SELECT COUNT(DISTINCT seller_id)

FROM olist_sellers_dataset

) AS Sellers;

""").df()

display(dashboard)

,Revenue,Orders,Customers,AvgReview,Sellers
0,16008872.12,99441,96096,4.09,3095


In [157]:
# ======================================================
# Top Customers by Orders
# ======================================================

con.sql("""

SELECT

    c.customer_unique_id,

    COUNT(DISTINCT o.order_id) AS TotalOrders

FROM olist_customers_dataset c

JOIN olist_orders_dataset o

ON c.customer_id = o.customer_id

GROUP BY c.customer_unique_id

ORDER BY TotalOrders DESC

LIMIT 20;

""").df()

,customer_unique_id,TotalOrders
0,8d50f5eadf50201ccdcedfb9e2ac8455,17
1,3e43e6105506432c953e165fb2acf44c,9
2,6469f99c1f9dfae7733b25662e7f1782,7
3,ca77025e7201e3b30c44b472ff346268,7
4,1b6c7548a2a1f9037c1fd3ddfed95f33,7
5,63cfc61cee11cbe306bff5857d00bfe4,6
6,dc813062e0fc23409cd255f7f53c7074,6
7,de34b16117594161a6a89c50b289d35a,6
8,47c1a3033b8b77b3ab6e109eb4d5fdf3,6
9,f0e310a6839dce9de1638e0fe5ab282a,6


In [158]:
# ======================================================
# Customer Segmentation
# ======================================================

con.sql("""

WITH CustomerRevenue AS (

SELECT

    c.customer_unique_id,

    SUM(p.payment_value) AS Revenue

FROM olist_customers_dataset c

JOIN olist_orders_dataset o

ON c.customer_id=o.customer_id

JOIN olist_order_payments_dataset p

ON o.order_id=p.order_id

GROUP BY c.customer_unique_id

)

SELECT

customer_unique_id,

Revenue,

CASE

WHEN Revenue>=1000 THEN 'High Value'

WHEN Revenue>=500 THEN 'Medium Value'

ELSE 'Low Value'

END AS CustomerSegment

FROM CustomerRevenue

ORDER BY Revenue DESC;

""").df()

,customer_unique_id,Revenue,CustomerSegment
0,0a0a92112bd4c708ca5fde585afaa872,13664.08,High Value
1,46450c74a0d8c5ca9395da1daac6c120,9553.02,High Value
2,da122df9eeddfedc1dc1f5349a1a690c,7571.63,High Value
3,763c8b1c9c68a0229c42c9fc6f662b93,7274.88,High Value
4,dc4802a71eae9be1dd28f5d788ceb526,6929.31,High Value
...,...,...,...
96090,b33336f46234b24a613ad9064d13106d,10.89,Low Value
96091,bd06ce0e06ad77a7f681f1a4960a3cc6,10.07,Low Value
96092,317cfc692e3f86c45c95697c61c853a6,9.59,Low Value
96093,4fa4365000c7090fcb8cad5713c6d3db,0.00,Low Value


In [159]:
# ======================================================
# Revenue Segment Summary
# ======================================================

con.sql("""

WITH CustomerRevenue AS (

SELECT

    c.customer_unique_id,

    SUM(p.payment_value) AS Revenue

FROM olist_customers_dataset c

JOIN olist_orders_dataset o

ON c.customer_id=o.customer_id

JOIN olist_order_payments_dataset p

ON o.order_id=p.order_id

GROUP BY c.customer_unique_id

)

SELECT

CASE

WHEN Revenue>=1000 THEN 'High Value'

WHEN Revenue>=500 THEN 'Medium Value'

ELSE 'Low Value'

END AS Segment,

COUNT(*) AS Customers,

ROUND(AVG(Revenue),2) AS AvgRevenue

FROM CustomerRevenue

GROUP BY Segment

ORDER BY AvgRevenue DESC;

""").df()

,Segment,Customers,AvgRevenue
0,High Value,1218,1600.10
1,Medium Value,3271,680.41
2,Low Value,91606,129.19


In [160]:
# ======================================================
# Seller Quartiles
# ======================================================

con.sql("""

SELECT

seller_id,

SUM(price) AS Revenue,

NTILE(4)

OVER(

ORDER BY SUM(price) DESC

) AS RevenueQuartile

FROM olist_order_items_dataset

GROUP BY seller_id;

""").df()

,seller_id,Revenue,RevenueQuartile
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1
1,53243585a1d6dc2643021fd1853d8905,222776.05,1
2,4a3ca9315b744ce9f8e9374361493884,200472.92,1
3,fa1c13f2614d7b5c4749cbc52fecda94,194042.03,1
4,7c67e1448b00f6e969d365cea6b010ab,187923.89,1
...,...,...,...
3090,34aefe746cd81b7f3b23253ea28bef39,8.00,4
3091,702835e4b785b67a084280efca355756,7.60,4
3092,1fa2d3def6adfa70e58c276bb64fe5bb,6.90,4
3093,77128dec4bec4878c37ab7d6169d6f26,6.50,4


In [161]:
# ======================================================
# Top Product per Category
# ======================================================

con.sql("""

WITH RankedProducts AS (

SELECT

t.product_category_name_english,

i.product_id,

SUM(i.price) AS Revenue,

ROW_NUMBER()

OVER(

PARTITION BY t.product_category_name_english

ORDER BY SUM(i.price) DESC

) AS rn

FROM olist_order_items_dataset i

JOIN olist_products_dataset p

ON i.product_id=p.product_id

JOIN product_category_name_translation t

ON p.product_category_name=t.product_category_name

GROUP BY

t.product_category_name_english,

i.product_id

)

SELECT *

FROM RankedProducts

WHERE rn=1;

""").df()

,product_category_name_english,product_id,Revenue,rn
0,baby,25c38557cf793876c5abdd5931f922db,38907.32,1
1,perfumery,595fac2a385ac33a80bd5114aec74eb8,12636.90,1
2,fashion_bags_accessories,d017a2151d543a9885604dc62a3d9dcc,6860.00,1
3,audio,13db47eae724e2848e12b71a617a3a41,12992.65,1
4,construction_tools_lights,a02d0123079f4ae96001ba2010d1a2df,13550.00,1
...,...,...,...,...
66,agro_industry_and_commerce,11250b0d4b709fee92441c5f34122aed,9111.00,1
67,fashio_female_clothing,63cf4d771cba1d380af927afe5895d4b,264.92,1
68,party_supplies,99b4596b82b6f75d453160c9c75e01f4,772.80,1
69,fashion_childrens_clothes,8cfc3506cedc0626364457d254429118,179.98,1


In [162]:
# ======================================================
# Running Customer Revenue
# ======================================================

con.sql("""

WITH CustomerRevenue AS (

SELECT

c.customer_unique_id,

SUM(p.payment_value) AS Revenue

FROM olist_customers_dataset c

JOIN olist_orders_dataset o

ON c.customer_id=o.customer_id

JOIN olist_order_payments_dataset p

ON o.order_id=p.order_id

GROUP BY c.customer_unique_id

)

SELECT

customer_unique_id,

Revenue,

SUM(Revenue)

OVER(

ORDER BY Revenue DESC

ROWS BETWEEN UNBOUNDED PRECEDING

AND CURRENT ROW

) AS RunningRevenue

FROM CustomerRevenue;

""").df()

,customer_unique_id,Revenue,RunningRevenue
0,0a0a92112bd4c708ca5fde585afaa872,13664.08,13664.08
1,46450c74a0d8c5ca9395da1daac6c120,9553.02,23217.10
2,da122df9eeddfedc1dc1f5349a1a690c,7571.63,30788.73
3,763c8b1c9c68a0229c42c9fc6f662b93,7274.88,38063.61
4,dc4802a71eae9be1dd28f5d788ceb526,6929.31,44992.92
...,...,...,...
96090,b33336f46234b24a613ad9064d13106d,10.89,16008852.46
96091,bd06ce0e06ad77a7f681f1a4960a3cc6,10.07,16008862.53
96092,317cfc692e3f86c45c95697c61c853a6,9.59,16008872.12
96093,4fa4365000c7090fcb8cad5713c6d3db,0.00,16008872.12


In [163]:
# ======================================================
# Revenue Comparison
# ======================================================

con.sql("""

WITH CustomerRevenue AS (

SELECT

c.customer_unique_id,

SUM(p.payment_value) AS Revenue

FROM olist_customers_dataset c

JOIN olist_orders_dataset o

ON c.customer_id=o.customer_id

JOIN olist_order_payments_dataset p

ON o.order_id=p.order_id

GROUP BY c.customer_unique_id

)

SELECT

customer_unique_id,

Revenue,

LAG(Revenue)

OVER(

ORDER BY Revenue DESC

) AS PreviousRevenue

FROM CustomerRevenue;

""").df()

,customer_unique_id,Revenue,PreviousRevenue
0,0a0a92112bd4c708ca5fde585afaa872,13664.08,NaN
1,46450c74a0d8c5ca9395da1daac6c120,9553.02,13664.08
2,da122df9eeddfedc1dc1f5349a1a690c,7571.63,9553.02
3,763c8b1c9c68a0229c42c9fc6f662b93,7274.88,7571.63
4,dc4802a71eae9be1dd28f5d788ceb526,6929.31,7274.88
...,...,...,...
96090,b33336f46234b24a613ad9064d13106d,10.89,11.63
96091,bd06ce0e06ad77a7f681f1a4960a3cc6,10.07,10.89
96092,317cfc692e3f86c45c95697c61c853a6,9.59,10.07
96093,4fa4365000c7090fcb8cad5713c6d3db,0.00,9.59


In [164]:
# ======================================================
# Next Revenue
# ======================================================

con.sql("""

WITH CustomerRevenue AS (

SELECT

c.customer_unique_id,

SUM(p.payment_value) AS Revenue

FROM olist_customers_dataset c

JOIN olist_orders_dataset o

ON c.customer_id=o.customer_id

JOIN olist_order_payments_dataset p

ON o.order_id=p.order_id

GROUP BY c.customer_unique_id

)

SELECT

customer_unique_id,

Revenue,

LEAD(Revenue)

OVER(

ORDER BY Revenue DESC

) AS NextRevenue

FROM CustomerRevenue;

""").df()

,customer_unique_id,Revenue,NextRevenue
0,0a0a92112bd4c708ca5fde585afaa872,13664.08,9553.02
1,46450c74a0d8c5ca9395da1daac6c120,9553.02,7571.63
2,da122df9eeddfedc1dc1f5349a1a690c,7571.63,7274.88
3,763c8b1c9c68a0229c42c9fc6f662b93,7274.88,6929.31
4,dc4802a71eae9be1dd28f5d788ceb526,6929.31,6922.21
...,...,...,...
96090,b33336f46234b24a613ad9064d13106d,10.89,10.07
96091,bd06ce0e06ad77a7f681f1a4960a3cc6,10.07,9.59
96092,317cfc692e3f86c45c95697c61c853a6,9.59,0.00
96093,4fa4365000c7090fcb8cad5713c6d3db,0.00,0.00


In [165]:
# ======================================================
# Revenue Contribution
# ======================================================

con.sql("""

SELECT

seller_id,

SUM(price) AS Revenue,

ROUND(

100.0 *

SUM(price)

/

SUM(SUM(price)) OVER(),

2

) AS RevenuePercent

FROM olist_order_items_dataset

GROUP BY seller_id

ORDER BY Revenue DESC

LIMIT 20;

""").df()

,seller_id,Revenue,RevenuePercent
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1.69
1,53243585a1d6dc2643021fd1853d8905,222776.05,1.64
2,4a3ca9315b744ce9f8e9374361493884,200472.92,1.47
3,fa1c13f2614d7b5c4749cbc52fecda94,194042.03,1.43
4,7c67e1448b00f6e969d365cea6b010ab,187923.89,1.38
5,7e93a43ef30c4f03f38b393420bc753a,176431.87,1.30
6,da8622b14eb17ae2831f4ac5b9dab84a,160236.57,1.18
7,7a67c85e85bb2ce8582c35f2203ad736,141745.53,1.04
8,1025f0e2d44d7041d6cf58b6550e0bfa,138968.55,1.02
9,955fee9216a65b617aa5c0531780ce60,135171.70,0.99


In [166]:
# ======================================================
# Executive SQL Dashboard
# ======================================================

con.sql("""

WITH KPI AS (

SELECT

(SELECT ROUND(SUM(payment_value),2)

FROM olist_order_payments_dataset) AS Revenue,

(SELECT COUNT(DISTINCT order_id)

FROM olist_orders_dataset) AS Orders,

(SELECT COUNT(DISTINCT customer_unique_id)

FROM olist_customers_dataset) AS Customers,

(SELECT COUNT(DISTINCT seller_id)

FROM olist_sellers_dataset) AS Sellers,

(SELECT ROUND(AVG(review_score),2)

FROM olist_order_reviews_dataset) AS AvgReview

)

SELECT * FROM KPI;

""").df()

,Revenue,Orders,Customers,Sellers,AvgReview
0,16008872.12,99441,96096,3095,4.09


In [167]:
# ======================================================
# ABC Customer Classification
# ======================================================

con.sql("""

WITH CustomerRevenue AS (

SELECT

    c.customer_unique_id,

    SUM(p.payment_value) AS Revenue

FROM olist_customers_dataset c

JOIN olist_orders_dataset o

ON c.customer_id = o.customer_id

JOIN olist_order_payments_dataset p

ON o.order_id = p.order_id

GROUP BY c.customer_unique_id

),

Ranked AS (

SELECT

*,

NTILE(3) OVER(

ORDER BY Revenue DESC

) AS CustomerClass

FROM CustomerRevenue

)

SELECT

customer_unique_id,

Revenue,

CASE

WHEN CustomerClass=1 THEN 'A'

WHEN CustomerClass=2 THEN 'B'

ELSE 'C'

END AS ABC_Class

FROM Ranked

ORDER BY Revenue DESC;

""").df()

,customer_unique_id,Revenue,ABC_Class
0,0a0a92112bd4c708ca5fde585afaa872,13664.08,A
1,46450c74a0d8c5ca9395da1daac6c120,9553.02,A
2,da122df9eeddfedc1dc1f5349a1a690c,7571.63,A
3,763c8b1c9c68a0229c42c9fc6f662b93,7274.88,A
4,dc4802a71eae9be1dd28f5d788ceb526,6929.31,A
...,...,...,...
96090,b33336f46234b24a613ad9064d13106d,10.89,C
96091,bd06ce0e06ad77a7f681f1a4960a3cc6,10.07,C
96092,317cfc692e3f86c45c95697c61c853a6,9.59,C
96093,968fac81e2c44fb6c1e3ac2a45e6a102,0.00,C


In [168]:
# ======================================================
# Pareto Analysis
# ======================================================

con.sql("""

WITH SellerRevenue AS (

SELECT

seller_id,

SUM(price) AS Revenue

FROM olist_order_items_dataset

GROUP BY seller_id

),

Running AS (

SELECT

seller_id,

Revenue,

SUM(Revenue) OVER(

ORDER BY Revenue DESC

ROWS BETWEEN UNBOUNDED PRECEDING

AND CURRENT ROW

) AS RunningRevenue,

SUM(Revenue) OVER() AS TotalRevenue

FROM SellerRevenue

)

SELECT

seller_id,

Revenue,

ROUND(

100*RunningRevenue/TotalRevenue,

2

) AS CumulativePercent

FROM Running

ORDER BY Revenue DESC;

""").df()

,seller_id,Revenue,CumulativePercent
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1.69
1,53243585a1d6dc2643021fd1853d8905,222776.05,3.33
2,4a3ca9315b744ce9f8e9374361493884,200472.92,4.80
3,fa1c13f2614d7b5c4749cbc52fecda94,194042.03,6.23
4,7c67e1448b00f6e969d365cea6b010ab,187923.89,7.61
...,...,...,...
3090,34aefe746cd81b7f3b23253ea28bef39,8.00,100.00
3091,702835e4b785b67a084280efca355756,7.60,100.00
3092,1fa2d3def6adfa70e58c276bb64fe5bb,6.90,100.00
3093,77128dec4bec4878c37ab7d6169d6f26,6.50,100.00


In [169]:
# ======================================================
# Average Order Value by State
# ======================================================

con.sql("""

SELECT

c.customer_state,

ROUND(

AVG(p.payment_value),

2

) AS AverageOrderValue

FROM olist_customers_dataset c

JOIN olist_orders_dataset o

ON c.customer_id=o.customer_id

JOIN olist_order_payments_dataset p

ON o.order_id=p.order_id

GROUP BY c.customer_state

ORDER BY AverageOrderValue DESC;

""").df()

,customer_state,AverageOrderValue
0,PB,248.33
1,AC,234.29
2,RO,233.20
3,AP,232.33
4,AL,227.08
5,RR,218.80
6,PA,215.92
7,SE,208.44
8,PI,207.11
9,TO,204.27


In [170]:
# ======================================================
# Customer Purchase Frequency
# ======================================================

con.sql("""

SELECT

c.customer_unique_id,

COUNT(DISTINCT o.order_id) AS Orders

FROM olist_customers_dataset c

JOIN olist_orders_dataset o

ON c.customer_id=o.customer_id

GROUP BY c.customer_unique_id

ORDER BY Orders DESC

LIMIT 20;

""").df()

,customer_unique_id,Orders
0,8d50f5eadf50201ccdcedfb9e2ac8455,17
1,3e43e6105506432c953e165fb2acf44c,9
2,6469f99c1f9dfae7733b25662e7f1782,7
3,1b6c7548a2a1f9037c1fd3ddfed95f33,7
4,ca77025e7201e3b30c44b472ff346268,7
5,dc813062e0fc23409cd255f7f53c7074,6
6,de34b16117594161a6a89c50b289d35a,6
7,12f5d6e1cbf93dafd9dcc19095df0b3d,6
8,47c1a3033b8b77b3ab6e109eb4d5fdf3,6
9,f0e310a6839dce9de1638e0fe5ab282a,6


In [171]:
# ======================================================
# Revenue by State
# ======================================================

con.sql("""

SELECT

c.customer_state,

ROUND(

SUM(p.payment_value),

2

) AS Revenue

FROM olist_customers_dataset c

JOIN olist_orders_dataset o

ON c.customer_id=o.customer_id

JOIN olist_order_payments_dataset p

ON o.order_id=p.order_id

GROUP BY c.customer_state

ORDER BY Revenue DESC

LIMIT 15;

""").df()

,customer_state,Revenue
0,SP,5998226.96
1,RJ,2144379.69
2,MG,1872257.26
3,RS,890898.54
4,PR,811156.38
5,SC,623086.43
6,BA,616645.82
7,DF,355141.08
8,GO,350092.31
9,ES,325967.55


In [172]:
# ======================================================
# Monthly Moving Average
# ======================================================

con.sql("""

WITH MonthlyRevenue AS (

SELECT

date_trunc(

'month',

CAST(order_purchase_timestamp AS TIMESTAMP)

) AS Month,

SUM(payment_value) AS Revenue

FROM olist_orders_dataset o

JOIN olist_order_payments_dataset p

ON o.order_id=p.order_id

GROUP BY Month

)

SELECT

Month,

Revenue,

ROUND(

AVG(Revenue)

OVER(

ORDER BY Month

ROWS BETWEEN 2 PRECEDING

AND CURRENT ROW

),

2

) AS MovingAverage

FROM MonthlyRevenue

ORDER BY Month;

""").df()

,Month,Revenue,MovingAverage
0,2016-09-01,252.24,252.24
1,2016-10-01,59090.48,29671.36
2,2016-12-01,19.62,19787.45
3,2017-01-01,138488.04,65866.05
4,2017-02-01,291908.01,143471.89
5,2017-03-01,449863.60,293419.88
6,2017-04-01,417788.03,386519.88
7,2017-05-01,592918.82,486856.82
8,2017-06-01,511276.38,507327.74
9,2017-07-01,592382.92,565526.04


In [173]:
# ======================================================
# Top Categories
# ======================================================

con.sql("""

SELECT

t.product_category_name_english,

ROUND(

SUM(i.price),

2

) AS Revenue

FROM olist_order_items_dataset i

JOIN olist_products_dataset p

ON i.product_id=p.product_id

JOIN product_category_name_translation t

ON p.product_category_name=t.product_category_name

GROUP BY t.product_category_name_english

ORDER BY Revenue DESC

LIMIT 10;

""").df()

,product_category_name_english,Revenue
0,health_beauty,1258681.34
1,watches_gifts,1205005.68
2,bed_bath_table,1036988.68
3,sports_leisure,988048.97
4,computers_accessories,911954.32
5,furniture_decor,729762.49
6,cool_stuff,635290.85
7,housewares,632248.66
8,auto,592720.11
9,garden_tools,485256.46


In [174]:
# ======================================================
# Revenue Share
# ======================================================

con.sql("""

SELECT

payment_type,

ROUND(

SUM(payment_value),

2

) AS Revenue,

ROUND(

100*SUM(payment_value)

/ SUM(SUM(payment_value)) OVER(),

2

) AS RevenueShare

FROM olist_order_payments_dataset

GROUP BY payment_type

ORDER BY Revenue DESC;

""").df()

,payment_type,Revenue,RevenueShare
0,credit_card,12542084.19,78.34
1,boleto,2869361.27,17.92
2,voucher,379436.87,2.37
3,debit_card,217989.79,1.36
4,not_defined,0.00,0.00


In [176]:
# ======================================================
# SQL KPI Dashboard
# ======================================================

con.sql("""

SELECT

COUNT(DISTINCT order_id) AS Orders,

ROUND(SUM(payment_value),2) AS Revenue,

ROUND(AVG(payment_value),2) AS AvgPayment,

MIN(payment_value) AS MinPayment,

MAX(payment_value) AS MaxPayment

FROM olist_order_payments_dataset;

""").df()

,Orders,Revenue,AvgPayment,MinPayment,MaxPayment
0,99440,16008872.12,154.1,0.0,13664.08


In [179]:
# ======================================================
# SQL Notebook Completed
# ======================================================

print("="*70)

print("Retail Intelligence Platform")

print("Notebook 04 Completed Successfully")

print("="*70)

print("Topics Covered")

print("- SQL Basics")
print("- JOIN")
print("- GROUP BY")
print("- HAVING")
print("- CTE")
print("- Window Functions")
print("- RANK")
print("- DENSE_RANK")
print("- ROW_NUMBER")
print("- NTILE")
print("- LAG")
print("- LEAD")
print("- Running Totals")
print("- Moving Average")
print("- Pareto Analysis")

print("="*70)

print("Next Notebook : 05_PowerBI_Data_Model.ipynb")

print("="*70)

Retail Intelligence Platform
Notebook 04 Completed Successfully
Topics Covered
- SQL Basics
- JOIN
- GROUP BY
- HAVING
- CTE
- Window Functions
- RANK
- DENSE_RANK
- ROW_NUMBER
- NTILE
- LAG
- LEAD
- Running Totals
- Moving Average
- Pareto Analysis
Next Notebook : 05_PowerBI_Data_Model.ipynb
